In [2]:
import uproot
import awkward as ak
import dask_awkward as dak
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings
from dask.diagnostics import ProgressBar
import zfit
from zfit.loss import UnbinnedNLL
from zfit.minimize import Minuit
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mplhep as hep

/home/tandrade/.local/lib/python3.9/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress the TensorFlow warnings AND this warning, set ZFIT_DISABLE_TF_WARNINGS=1.
  warnings.warn(


In [ ]:
tree_path = "Events"
data_files = {f: tree_path for f in [
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022C.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022D.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022E.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022F.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022G.root"
]}

In [ ]:
def apply_selection(df):

    kstar_pdg = 0.892
    mask_kpi_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) <= 0.150
    mask_pik_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg) <= 0.150
    is_kpi_closer = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) < \
                    abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg)
    
    mask = (
        (((df['BToTrkTrkMuMu_mll_fullfit'] > 1.0) & (df['BToTrkTrkMuMu_mll_fullfit'] < 2.7))  | 
         ((df['BToTrkTrkMuMu_mll_fullfit'] > 4.0) & (df['BToTrkTrkMuMu_mll_fullfit'] < 6.0))) &
        
        (mask_kpi_window | mask_pik_window) &         
        (is_kpi_closer) &
        (df['BPH_1Muon_pt'] > 2.0) & 
        (df['BPH_2Muon_pt'] > 2.0) & 
        (abs(df['BPH_1Muon_eta']) < 2.4) & 
        (abs(df['BPH_2Muon_eta']) < 2.4) &
        (df['BToTrkTrkMuMu_fit_trk1_pt'] > 1.5) &
        (df['BToTrkTrkMuMu_fit_trk2_pt'] > 1.5) 
    )
    
    cols_to_keep = [
        'BToTrkTrkMuMu_fit_mass_Kpi', "BToTrkTrkMuMu_mll_fullfit"
    ]
        
    return df[cols_to_keep][mask]

In [ ]:
def build_dataframe(file_dict, label="Dataset"):
    print(f"\nProcessando {label}...")

    df = uproot.dask(file_dict)
    df = apply_selection(df)

    with ProgressBar():
        awkward_array = df.compute()
        df_pandas = ak.to_dataframe(awkward_array).reset_index(drop=True)

    initial_len = len(df_pandas)
    df_pandas = df_pandas.replace([np.inf, -np.inf], np.nan)
    df_pandas = df_pandas.dropna()
    final_len  = len(df_pandas)
    removed    = initial_len - final_len

    if removed > 0:
        print(f"  [Limpeza] Removidos {removed} eventos com NaN ou Inf ({removed/initial_len:.2%})")

    print(f"{label} finalizado. Linhas: {final_len}")
    return df_pandas

In [ ]:
df_data = build_dataframe(data_files, "Data")

print(f"\nEventos de background (sidebands): {len(df_data)}")

In [ ]:
obs = zfit.Space("BToTrkTrkMuMu_fit_mass_Kpi", limits=(5.0, 5.6))
data_zfit = zfit.Data.from_pandas(df_data["BToTrkTrkMuMu_fit_mass_Kpi"], obs=obs)

In [ ]:
mu    = zfit.Parameter("mu", 5.28, 5.15, 5.40)
sigma = zfit.Parameter("sigma", 0.025, 0.005, 0.100)
alphal = zfit.Parameter("alphal", 1.5, 0.1, 5.0)
nl     = zfit.Parameter("nl", 5.0, 0.1, 100.0)
alphar = zfit.Parameter("alphar", 1.5, 0.1, 5.0)
nr     = zfit.Parameter("nr", 5.0, 0.1, 30.0)

In [ ]:
tau = zfit.Parameter("tau", -2.0, -10.0, 0.0)

In [ ]:
total_ev = len(df_data)
n_sig = zfit.Parameter("n_sig", total_ev * 0.1, 0, total_ev)
n_bkg = zfit.Parameter("n_bkg", total_ev * 0.9, 0, total_ev)

In [ ]:
signal_pdf = zfit.pdf.DoubleCB(mu=mu, sigma=sigma, alphal=alphal, nl=nl, alphar=alphar, nr=nr, obs=obs)
bkg_pdf    = zfit.pdf.Exponential(lambda_=tau, obs=obs)

signal_ext = signal_pdf.create_extended(n_sig)
bkg_ext    = bkg_pdf.create_extended(n_bkg)
model = zfit.pdf.SumPDF([signal_ext, bkg_ext])

In [ ]:
nll = zfit.loss.ExtendedUnbinnedNLL(model, data_zfit)
minimizer = zfit.minimize.Minuit(tol=1e-3)
result = minimizer.minimize(nll)

In [ ]:
print(result)

In [ ]:
plt.figure(figsize=(8, 6))
counts, edges = np.histogram(df_data["BToTrkTrkMuMu_fit_mass_Kpi"], bins=50, range=(5.0, 5.6))
hep.histplot(counts, bins=edges, yerr=True, histtype='errorbar', color='black', label='Dados')

x_plot = np.linspace(5.0, 5.6, 1000)
bw = 0.012 
plt.plot(x_plot, model.ext_pdf(x_plot).numpy() * bw, color='blue', lw=3, label='Fit Total')
plt.plot(x_plot, signal_ext.ext_pdf(x_plot).numpy() * bw, color='red', ls='--', label='Sinal (Double CB)')
plt.plot(x_plot, bkg_ext.ext_pdf(x_plot).numpy() * bw, color='green', ls=':', label='Fundo (Exp)')

plt.xlabel(r"$m(K \pi \mu \mu)$ [GeV]", fontsize = 15)
plt.ylabel(f"Events / {bw:.3f} GeV", fontsize = 15)
plt.xlim(5.0, 5.6)
plt.legend(fontsize=15)
plt.tick_params(axis='both', which='major', labelsize=16)
plt.show()